# Female ICU Blood Disease Survivors With Prescribed Medications

This Google Colab notebook runs the attached BigQuery SQL query against the MIMIC-III clinical dataset and downloads the output as a `.csv` file.

**Cohort definition**

- Female ICU patients
- ICD-9 diagnosis codes between `280–289`
- Includes Survivors and Non Survivors
- Includes only rows where medications were prescribed
- Removes duplicate drug rows using `SELECT DISTINCT`

Replace `YOUR_PROJECT_ID_HERE` with your actual Google Cloud Project ID before running the query.


## 1. Install and import required libraries


In [10]:
# Install required Google Cloud BigQuery libraries
!pip -q install google-cloud-bigquery pandas pyarrow

# Import Python libraries
import pandas as pd
from google.colab import auth, files
from google.cloud import bigquery


## 2. Authenticate Google Cloud access

When prompted, sign in with a Google account that has access to your Google Cloud project and BigQuery.


In [11]:
auth.authenticate_user()
print("Authentication complete.")


Authentication complete.


## 3. Steps to Execute the code block below:
Set Your Google Cloud Project for Big Query Client.

Use the following steps to get the project ID

1. Navigate to console.cloud.google.com
2. Copy the project ID and replace the value for the tag PROJECT_ID below.


In [12]:
PROJECT_ID = 'project-4044d9a5-d0d2-4d71-b82'

if PROJECT_ID == "YOUR_PROJECT_ID_HERE" or not PROJECT_ID.strip():
    raise ValueError("Please replace PROJECT_ID with your actual Google Cloud Project ID.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
client = bigquery.Client(project=PROJECT_ID)

print("Using BigQuery project:", client.project)


Using BigQuery project: project-4044d9a5-d0d2-4d71-b82


## 4. BigQuery SQL

This is the SQL from your uploaded file.


In [14]:
query = r"""
-- ============================================================
-- Female ICU patients with blood disease ICD-9 codes 280–289
-- Include BOTH survivors and non-survivors
-- Include only rows where medications were prescribed
-- Remove duplicate drug rows
-- ============================================================

WITH blood_disease_admissions AS (
  SELECT DISTINCT
      subject_id,
      hadm_id,
      icd9_code
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
  WHERE icd9_code BETWEEN '280' AND '289'
),

female_icu_patients AS (
  SELECT DISTINCT
      icu.subject_id,
      icu.hadm_id,
      icu.icustay_id,
      pat.gender,
      adm.admittime,
      adm.dischtime,
      adm.deathtime,
      adm.hospital_expire_flag,

      -- Mortality label:
      -- 0 = survived hospital admission
      -- 1 = died during hospital admission
      CASE
        WHEN adm.hospital_expire_flag = 1 THEN 1
        ELSE 0
      END AS mortality,

      icu.intime,
      icu.outtime
  FROM `physionet-data.mimiciii_clinical.icustays` icu
  JOIN `physionet-data.mimiciii_clinical.patients` pat
      ON icu.subject_id = pat.subject_id
  JOIN `physionet-data.mimiciii_clinical.admissions` adm
      ON icu.hadm_id = adm.hadm_id
  JOIN blood_disease_admissions bd
      ON icu.hadm_id = bd.hadm_id
  WHERE pat.gender = 'F'
),

dedup_medications AS (
  SELECT DISTINCT
      subject_id,
      hadm_id,
      LOWER(TRIM(drug)) AS drug,
      LOWER(TRIM(drug_name_generic)) AS drug_name_generic,
      route
  FROM `physionet-data.mimiciii_clinical.prescriptions`
  WHERE drug IS NOT NULL
)

SELECT DISTINCT
    fp.subject_id,
    fp.hadm_id,
    fp.icustay_id,
    fp.gender,
    fp.admittime,
    fp.dischtime,
    fp.deathtime,
    fp.hospital_expire_flag,
    fp.mortality,
    fp.intime,
    fp.outtime,

    bd.icd9_code,

    med.drug,
    med.drug_name_generic,
    med.route

FROM female_icu_patients fp

JOIN blood_disease_admissions bd
    ON fp.hadm_id = bd.hadm_id

JOIN dedup_medications med
    ON fp.hadm_id = med.hadm_id

ORDER BY
    fp.subject_id,
    fp.hadm_id,
    med.drug;
"""
print(query[:1000])



-- ============================================================
-- Female ICU patients with blood disease ICD-9 codes 280–289
-- Include BOTH survivors and non-survivors
-- Include only rows where medications were prescribed
-- Remove duplicate drug rows
-- ============================================================

WITH blood_disease_admissions AS (
  SELECT DISTINCT
      subject_id,
      hadm_id,
      icd9_code
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
  WHERE icd9_code BETWEEN '280' AND '289'
),

female_icu_patients AS (
  SELECT DISTINCT
      icu.subject_id,
      icu.hadm_id,
      icu.icustay_id,
      pat.gender,
      adm.admittime,
      adm.dischtime,
      adm.deathtime,
      adm.hospital_expire_flag,

      -- Mortality label:
      -- 0 = survived hospital admission
      -- 1 = died during hospital admission
      CASE
        WHEN adm.hospital_expire_flag = 1 THEN 1
        ELSE 0
      END AS mortality,

      icu.intime,
      icu.outtime
  FROM `

## 5. Run the query and load results into a DataFrame


In [15]:
query_job = client.query(query)
df = query_job.to_dataframe()

print("Query complete.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


Query complete.
Rows: 633247
Columns: 15


,subject_id,hadm_id,icustay_id,gender,admittime,dischtime,deathtime,hospital_expire_flag,mortality,intime,outtime,icd9_code,drug,drug_name_generic,route
0,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,1/2 ns,None,IV
1,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,acetaminophen,acetaminophen,PO
2,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,albuterol,albuterol inhaler,IH
3,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,amlodipine,amlodipine,PO
4,6,107064,228232,F,2175-05-30 07:15:00,2175-06-15 16:00:00,NaT,0,0,2175-05-30 21:30:54,2175-06-03 13:39:54,2859,anti-thymocyte globulin (rabbit),None,IV


## 6. Basic validation summary


In [16]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))

print("\nUnique patients:", df["subject_id"].nunique() if "subject_id" in df.columns else "subject_id not found")
print("Unique admissions:", df["hadm_id"].nunique() if "hadm_id" in df.columns else "hadm_id not found")
print("Unique ICU stays:", df["icustay_id"].nunique() if "icustay_id" in df.columns else "icustay_id not found")
print("Unique drugs:", df["drug"].nunique() if "drug" in df.columns else "drug not found")


Dataset shape: (633247, 15)

Columns:
['subject_id', 'hadm_id', 'icustay_id', 'gender', 'admittime', 'dischtime', 'deathtime', 'hospital_expire_flag', 'mortality', 'intime', 'outtime', 'icd9_code', 'drug', 'drug_name_generic', 'route']

Missing values per column:


,missing_count
subject_id,0
hadm_id,0
icustay_id,0
gender,0
admittime,0
dischtime,0
deathtime,518239
hospital_expire_flag,0
mortality,0
intime,0



Unique patients: 6944
Unique admissions: 8504
Unique ICU stays: 9341
Unique drugs: 2199


## 7. Save output as CSV


In [17]:
OUTPUT_CSV = "female_icu_blood_disease_with_medications.csv"

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV file: {OUTPUT_CSV}")


Saved CSV file: female_icu_blood_disease_with_medications.csv


## 8. Download the CSV file to your computer


In [18]:
files.download(OUTPUT_CSV)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>